# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --python python3.13 --seed --clear

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter accelerate>=0.26.0

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

downloading uv 0.11.7 x86_64-unknown-linux-gnu
installing to /home/potato/.local/bin
  uv
  uvx
everything's installed!
Using CPython 3.13.13
Creating virtual environment with seed packages at: .venv
 + pip==26.0.1
Activate with: source .venv/bin/activate
Installed kernelspec cse151b in /home/potato/.local/share/jupyter/kernels/cse151b
Done. Restart the kernel before proceeding.
Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.


### Run the cell below every time to activate the installed environment. 

In [1]:
# activate venv after installation. This needs to be run everytime.
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

!source ./.venv/bin/activate

Installed kernelspec cse151b in /home/potato/.local/share/jupyter/kernels/cse151b


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [2]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("CUDA Device Count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")
else:
    print("\n⚠️  No CUDA GPUs detected!")
    print("Check your CUDA installation or install PyTorch with CUDA support:")
    print("pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")

CUDA Available: True
CUDA Device Count: 1
GPU 0: NVIDIA GeForce RTX 4060 Laptop GPU
  Memory: 8.00 GB


In [3]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 4096 #32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [4]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [5]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Before you start to calculate, write down your reasoning and the steps you will take to solve the problem. "
    "Do not change your reasoning after you start calculating, unless there is a serious error. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

In [7]:
llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.85,
    max_model_len=16384/2,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
    # enable_lora=True,     # <-------------------- Enable/Disable LoRA support
    max_lora_rank=64,
)

# import transformers

# llm = transformers.AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     quantization_config=transformers.BitsAndBytesConfig(
#         load_in_4bit=True,
#         bnb_4bit_compute_dtype=torch.float16,
#         bnb_4bit_quant_type="nf4",
#         bnb_4bit_use_double_quant=True,
#     ),
#     device_map="auto",
#     trust_remote_code=True,
# ).to("cuda")


print("Model loaded.")

INFO 04-26 14:55:01 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 8192.0, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'max_num_batched_tokens': 32768, 'max_num_seqs': 256, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'max_lora_rank': 64, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


INFO 04-26 14:55:01 [model.py:549] Resolved architecture: Qwen3ForCausalLM
INFO 04-26 14:55:01 [model.py:1678] Using max model len 8192
INFO 04-26 14:55:01 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=32768.
INFO 04-26 14:55:03 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-26 14:55:04 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized; WSL is detected and NVML is not compatible with fork
(EngineCore pid=519098) INFO 04-26 14:55:09 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_s

(EngineCore pid=519098) <frozen importlib._bootstrap_external>:1325: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=519098) <frozen importlib._bootstrap_external>:1325: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


(EngineCore pid=519098) INFO 04-26 14:55:12 [bitsandbytes_loader.py:786] Loading weights with BitsAndBytes quantization. May take a while ...


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:08<00:16,  8.16s/it]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:14<00:06,  6.86s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:14<00:00,  3.79s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:14<00:00,  4.75s/it]
(EngineCore pid=519098) 


(EngineCore pid=519098) INFO 04-26 14:55:28 [gpu_model_runner.py:4820] Model loading took 2.7 GiB memory and 16.540471 seconds
(EngineCore pid=519098) INFO 04-26 14:55:31 [backends.py:1051] Using cache directory: /home/potato/.cache/vllm/torch_compile_cache/8c9eadd05f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=519098) INFO 04-26 14:55:31 [backends.py:1111] Dynamo bytecode transform time: 3.58 s
(EngineCore pid=519098) INFO 04-26 14:55:33 [backends.py:285] Directly load the compiled graph(s) for compile range (1, 32768) from the cache, took 1.017 s
(EngineCore pid=519098) INFO 04-26 14:55:33 [decorators.py:305] Directly load AOT compilation from path /home/potato/.cache/vllm/torch_compile_cache/torch_aot_compile/32c68858b848ac99ab8fb125b243dde8059a1f7f658e5875f8224acdc63ade58/rank_0_0/model
(EngineCore pid=519098) INFO 04-26 14:55:33 [monitor.py:48] torch.compile took 4.87 s in total
(EngineCore pid=519098) INFO 04-26 14:55:34 [monitor.py:76] Initial profiling/warmup run

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:10<00:00,  5.00it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:06<00:00,  5.83it/s]


(EngineCore pid=519098) INFO 04-26 14:56:04 [gpu_model_runner.py:6046] Graph capturing finished in 17 secs, took 0.96 GiB
(EngineCore pid=519098) INFO 04-26 14:56:04 [gpu_worker.py:597] CUDA graph pool memory: 0.96 GiB (actual), 0.99 GiB (estimated), difference: 0.03 GiB (2.6%).
(EngineCore pid=519098) INFO 04-26 14:56:04 [core.py:283] init engine (profile, create kv cache, warmup model) took 36.82 seconds
(EngineCore pid=519098) INFO 04-26 14:56:07 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-26 14:56:07 [interface.py:525] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
Model loaded.


In [8]:
print(llm)

Qwen3ForCausalLM()(
  (model): Qwen3Model()(
    (embed_tokens): VocabParallelEmbedding(num_embeddings=151936, embedding_dim=2560, org_vocab_size=151936, num_embeddings_padded=151936, tp_size=1)
    (layers): ModuleList()(
      (0-35): 36 x Qwen3DecoderLayer()(
        (self_attn): Qwen3Attention()(
          (qkv_proj): QKVParallelLinear(quant=BitsAndBytesLinearMethod, in_features=2560, output_features=6144, bias=False, tp_size=1, gather_output=False)
          (o_proj): RowParallelLinear(quant=BitsAndBytesLinearMethod, in_features=4096, output_features=2560, bias=False, tp_size=1, reduce_results=True)
          (rotary_emb): RotaryEmbedding(head_size=128, rotary_dim=128, max_position_embeddings=262144, base=5000000, is_neox_style=True)(
            (apply_rotary_emb): ApplyRotaryEmb(is_neox_style=True, enable_fp32_compute=False)
          )
          (attn): Attention(head_size=128, num_heads=32, num_kv_heads=8, scale=0.08838834764831845, backend=FlashAttentionImpl)
          (q_nor

In [9]:
sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [10]:
from vllm.lora.request import LoRARequest

# Build prompts for first 20 entries
prompts = []
for item in data[:20]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(
    prompts, 
    sampling_params=sampling_params,
    # lora_request=LoRARequest("math_finetune", 1, lora_path="./lora_adapter_comp/lora_adapter_comp"),
    # lora_request=LoRARequest("math_finetune", 1, lora_path="./lora_adapters/lora_adapter_openr1_generate/lora_adapter_openr1_generate"),
)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# responses = []
# print(f"Generating responses for {len(prompts)} questions...")
# for prompt in prompts:
#     inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)

#     with torch.no_grad():
#         output = llm.generate(
#             **inputs,
#             max_new_tokens=MAX_TOKENS,
#             temperature=0.6,
#             top_p=0.95,
#             top_k=20,
#             do_sample=True,
#             use_cache=True,
#         )

#     text = tokenizer.decode(output[0], skip_special_tokens=True)
#     responses.append(text)
#     print("Responded to a prompt!")

Generating responses for 20 questions...


Rendering prompts:   0%|          | 0/20 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/20 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


── Response 0 (id=0) ──
Okay, let's see. I need to find the sum of the first 325 positive even whole numbers. Hmm, first, let me make sure I know what the first positive even whole numbers are. Positive even whole numbers start from 2, right? So 2, 4, 6, 8, ..., up to the 325th one. 

Wait, the problem says "the first 325 positive even whole numbers". So the first one is 2, the second is 4, ..., the nth one is 2n. Let me ...

── Response 1 (id=1) ──
Okay, let's try to solve this integral. The problem is the integral from negative infinity to positive infinity of (a^(3/2)) divided by (s^2 + a^2) ds. Hmm, first, I need to check if this integral converges. Since the integrand is a function of s, and the limits are from -infty to +infty, it's an improper integral. Let me recall that integrals of the form 1/(s^2 + b^2) over the real line converge  ...

── Response 2 (id=2) ──
Okay, let's try to solve this problem step by step. First, part (a) and (b) are both about Newton's Law of Cooling,

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [11]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   2%|▏         | 20/1126 [00:00<00:44, 24.59it/s]

Scoring complete. 20 results.


## 8. Summary

Print accuracy broken down by question type.

In [12]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    2 /    9  (22.22%)
  Free-form  :    6 /   11  (54.55%)
  Overall    :    8 /   20  (40.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [13]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 20 records to results/starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!

## LoRA Training

In [4]:
!.venv/bin/python -m pip install peft trl pandas datasets

  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
  Using cached trl-1.2.0-py3-none-any.whl.metadata (11 kB)
  Using cached pandas-3.0.2-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached pyarrow-24.0.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached xxhash-3.6.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
Using cached peft-0.19.1-py3-none-any.whl (680 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 5.0 MB/s  0:00:00
Using cached pandas-3.0.2-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
Using cached datasets-4.8.4-py3-none-any.whl (526 kB)
Using cached pyarrow-24.0.0-cp313-cp313-manylinux_2_28_x86_64.whl (48.9 MB)
Using cached xxhash-3.6.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64

In [6]:
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig, TaskType
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import pandas as pd
from datasets import Dataset, load_dataset, concatenate_datasets
from huggingface_hub import hf_hub_download
from trl import SFTTrainer, SFTConfig
import re

In [7]:
# Do not change
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)

# Do not change
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# Can change r and lora_alpha as needed, r is rank and lora_alpha is a scaling factor. 
# Higher values of lora_alpha can lead to better performance but require more training time and resources.
# Usually lora_alpha = r * 2, although not required.
# Common: r=64, lora_alpha=128
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["qkv_proj", "o_proj", "gate_up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

trainable params: 21,823,488 || all params: 4,044,291,584 || trainable%: 0.5396


### qwedsacf/competition_math Dataset
Not very good, only free response questions

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# Load the Parquet file directly into Pandas
file_path = hf_hub_download(
    repo_id="qwedsacf/competition_math", 
    filename="data/train-00000-of-00001-7320a6f3aba8ebd2.parquet",
    repo_type="dataset",
)
ds = pd.read_parquet(file_path)
ds = ds[ds["solution"].str.contains(r"\\boxed\{(.+?)\}", na=False)] # Ignore the warning
ds = ds.reset_index(drop=True)

# Add the lora training prompt formatting
def lora_thinking_prompt(example):
    system = "You are a helpful assistant that thinks step-by-step."
    user = example["problem"]
    solution = example["solution"]
    answer = re.search(r"\\boxed\{(.+?)\}", solution).group()
    
    assistant_content = f"<think>\n{solution}\n</think>\nFinal Answer: {answer}"
    
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant_content}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

# Process and convert to HF Dataset for training
ds["text"] = ds.apply(lora_thinking_prompt, axis=1)

lora_train_dataset = Dataset.from_pandas(ds[["text"]])

/tmp/ipykernel_217574/2535021671.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ds = ds[ds["solution"].str.contains(r"\\boxed\{(.+?)\}", na=False)] # Ignore the warning


### open-r1/OpenR1-Math-220k Dataset
Has both free response and multiple choice

In [8]:
# Number of cores for parallel processing - adjust based on your machine
CORES = 8

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# Helper function to filter out samples without valid reasoning and correct math verification
def has_valid_reasoning(example):
    for comp, ver in zip(example["is_reasoning_complete"], example["correctness_math_verify"]):
        if comp == True and ver == True:
            return True
    return False

# Load the Dataset directly using Hugging Face Datasets library
ds = load_dataset("open-r1/OpenR1-Math-220k", "default") # Total dataset is 93.7k
# Pick first 3000 MCQ samples
mcq_ds = ds["train"].filter(
    lambda x: x["question_type"] == "MCQ" and has_valid_reasoning(x) and x["answer"].isalpha(), 
    num_proc=CORES
).select(range(3000)) 
# Pick first 7000 word problem samples
word_prob_ds = ds["train"].filter(
    lambda x: x["question_type"] == "math-word-problem" and has_valid_reasoning(x), 
    num_proc=CORES
).select(range(7000)) 
ds = concatenate_datasets([mcq_ds, word_prob_ds]) # Combine for 10k total samples

# Add the lora training prompt formatting
def lora_thinking_prompt_map(example):
    question_type = example["question_type"]
    system = SYSTEM_PROMPT_MCQ if question_type == "MCQ" else SYSTEM_PROMPT_MATH

    user = example["problem"]

    generations = example["generations"]
    is_reasoning_complete = example["is_reasoning_complete"]
    correctness_math_verify = example["correctness_math_verify"]
    solution = None
    for gen, comp, ver in zip(generations, is_reasoning_complete, correctness_math_verify):
        if comp == True and ver == True:
            solution = gen
            break
    if solution is None:
        return {"text": None}

    answer = f"\\boxed{{{example['answer']}}}"

    assistant_content = f"<think>\n{solution}\n</think>\nFinal Answer: {answer}"
    
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant_content}
    ]
    
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

# Process the dataset
lora_train_dataset = ds.map(
    lora_thinking_prompt_map,
    remove_columns=ds.column_names,
    num_proc=CORES
)

lora_train_dataset = lora_train_dataset.filter(
    lambda x: x["text"] is not None, 
    num_proc=CORES
)
lora_train_dataset = lora_train_dataset.shuffle(seed=42)

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

In [9]:
print(len(lora_train_dataset))
print(lora_train_dataset[0])
print(lora_train_dataset[1])
print(lora_train_dataset[2])
print(lora_train_dataset[3])
print(lora_train_dataset[4])

10000
{'text': '<|im_start|>system\nYou are an expert mathematician. Solve the problem step-by-step. Before you start to calculate, write down your reasoning and the steps you will take to solve the problem. Do not change your reasoning after you start calculating, unless there is a serious error. Put your final answer inside \\boxed{}. If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, e.g. \\boxed{3, 7}.<|im_end|>\n<|im_start|>user\nProblem 1. Buratino, Karabas-Barabas, and Duremar are running along a path around a circular pond. They start simultaneously from the same point, with Buratino running in one direction and Karabas-Barabas and Duremar running in the opposite direction. Buratino runs three times faster than Duremar and four times faster than Karabas-Barabas. After Buratino meets Duremar, he meets Karabas-Barabas 150 meters further. What is the length of the path around the pond?<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, let\'s

In [10]:
# CHECKPOINT_DIR = "./lora_adapters/lora_adapter_comp/checkpoints"
# MODEL_DIR = "./lora_adapters/lora_adapter_comp/lora_adapter_comp"

CHECKPOINT_DIR = "./lora_adapters/lora_adapter_openr1_generate/checkpoints"
MODEL_DIR = "./lora_adapters/lora_adapter_openr1_generate/lora_adapter_openr1_generate"

# Can change these parameters as needed
sft_config = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    max_length=4096, # Samples can be up to 3k tokens, use at leask 4k to avoid truncation.
    dataset_text_field="text",
    packing=False,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=50,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=True,
    optim="adamw_8bit",
    num_train_epochs=1,
    save_strategy="steps",
    save_steps=10,
)

# Do not change
trainer = SFTTrainer(
    model = model,
    train_dataset = lora_train_dataset,
    args=sft_config,
)

trainer.train()

trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.694409
20,0.569795
30,0.495855
40,0.452966
50,0.430109
60,0.434697
70,0.427057
80,0.418076
90,0.425521
100,0.415645


('./lora_adapters/lora_adapter_openr1_generate/lora_adapter_openr1_generate/tokenizer_config.json',
 './lora_adapters/lora_adapter_openr1_generate/lora_adapter_openr1_generate/chat_template.jinja',
 './lora_adapters/lora_adapter_openr1_generate/lora_adapter_openr1_generate/tokenizer.json')